# FoodChain AI - Model Training (K-Means)

## Objective
Train a K-Means model to cluster suppliers based on engineered features and build a recommendation-ready dataset.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
import joblib

df = pd.read_csv('../data/pune_supplier_dataset.csv')
# Fix: calculate distance and filter delivery radius to match Notebook 02
from math import radians, sin, cos, sqrt, atan2
def haversine_distance(lat1, lon1, lat2, lon2):
    R = 6371
    dlat = radians(lat2-lat1)
    dlon = radians(lon2-lon1)
    a = sin(dlat/2)**2 + cos(radians(lat1))*cos(radians(lat2))*sin(dlon/2)**2
    c = 2*atan2(sqrt(a), sqrt(1-a))
    return R*c

df['distance_km'] = df.apply(lambda x: haversine_distance(18.5204, 73.8567, x['latitude'], x['longitude']), axis=1)
df = df[df['distance_km'] <= df['delivery_radius_km']].copy()

X = pd.read_csv('../data/processed_features.csv')
X.head()


## Find Optimal Number of Clusters

In [ ]:
inertia=[]
scores=[]

for k in range(2,11):
    model=KMeans(n_clusters=k,random_state=42,n_init=10)
    labels=model.fit_predict(X)
    inertia.append(model.inertia_)
    scores.append(silhouette_score(X,labels))

plt.figure(figsize=(7,4))
plt.plot(range(2,11),inertia,marker='o')
plt.title('Elbow Method')
plt.xlabel('Clusters')
plt.ylabel('Inertia')
plt.grid()
plt.show()

plt.figure(figsize=(7,4))
plt.plot(range(2,11),scores,marker='o')
plt.title('Silhouette Score')
plt.xlabel('Clusters')
plt.ylabel('Score')
plt.grid()
plt.show()

best_k=range(2,11)[np.argmax(scores)]
print('Best K =',best_k)


## Train Final K-Means Model

In [ ]:
kmeans=KMeans(n_clusters=best_k,random_state=42,n_init=10)
df['cluster']=kmeans.fit_predict(X)

df[['supplier_name','ingredient','cluster']].head()


## Cluster Analysis

In [ ]:
cluster_summary=df.groupby('cluster')[[
'price',
'distance_km',
'rating',
'quality_score',
'reliability_score',
'average_delivery_time_min'
]].mean()

cluster_summary


## Recommendation Score

In [ ]:
weights={
'distance_km':0.35,
'price':0.30,
'rating':0.15,
'quality_score':0.10,
'reliability_score':0.05,
'average_delivery_time_min':0.05
}

recommendation_df=df.copy()

recommendation_df['recommendation_score']=(
-weights['distance_km']*recommendation_df['distance_km']
-weights['price']*recommendation_df['price']
+weights['rating']*recommendation_df['rating']
+weights['quality_score']*recommendation_df['quality_score']
+weights['reliability_score']*recommendation_df['reliability_score']
-weights['average_delivery_time_min']*recommendation_df['average_delivery_time_min']
)

recommendation_df.sort_values('recommendation_score',ascending=False).head(10)


## Save Model

In [ ]:
import os
os.makedirs('../models',exist_ok=True)

joblib.dump(kmeans,'../models/kmeans_model.pkl')
recommendation_df.to_csv('../data/supplier_clustered.csv',index=False)

print('Model and clustered dataset saved.')


# Key Observations

- Optimal K selected using Elbow and Silhouette methods.
- Suppliers segmented into business-friendly clusters.
- Recommendation score combines clustering with business priorities.
- Model saved for FastAPI inference.
- Clustered dataset will power the recommendation engine.
